# Semantic Business Layer for ManagedService-X

Builds a curated **business-entity (gold) layer** over `ManagedServiceData` so AI agents
(Fabric IQ / Foundry IQ) reason over **business objects and relationships** instead of raw tables.

**Entities produced (prefix `sem_`):**

| Table | Grain | Represents |
|-------|-------|------------|
| `sem_dim_customer` | one row per customer/tenant | Customer/Tenant (Inforcer posture + CRM account) |
| `sem_fact_assessment` | one assessment run | Assessment (Security, Copilot Readiness, Copilot Assessment) |
| `sem_fact_check` | one check result | Check / control finding |
| `sem_fact_posture` | one row per customer | Security posture snapshot + exposure |
| `sem_dim_recommendation` | catalog row | Recommendation catalog (SKU talking points) |
| `sem_fact_recommendation` | customer × rule | Managed-service opportunity (failed checks → recommended SKU) |

**Relationships:** `Customer 1—* Assessment`, `Assessment 1—* Check`, `Customer 1—1 Posture`,
`Customer 1—* Recommendation-opportunity`, `Recommendation-opportunity *—1 Recommendation-catalog`.

Join key across facts = `customer_key` (normalized tenant name); checks/assessments also share `assessment_id`.


In [20]:
# Semantic Business Layer — setup & helpers
from pyspark.sql import functions as F, Window

# Default lakehouse: ManagedServiceData (schema-enabled, schema = dbo)
SCHEMA = "dbo"

def rd(table_name):
    """Read a source/gold Delta table from the default lakehouse (dbo schema)."""
    return spark.table(f"{SCHEMA}.{table_name}")

def write_gold(df, table_name):
    """Overwrite a curated semantic-layer table in the dbo schema."""
    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .format("delta")
       .saveAsTable(f"{SCHEMA}.{table_name}"))
    print(f"wrote {SCHEMA}.{table_name}: {df.count():,} rows, {len(df.columns)} cols")

def norm_key(col):
    """Normalize a tenant/customer name into a stable join key."""
    return F.lower(F.trim(col))

# Common business-entity suffixes stripped by name_short()
_BIZ_SUFFIX = (
    r"(?i)\s+"
    r"(ltd\.?|limited|inc\.?|incorporated|corp\.?|corporation|"
    r"group|holdings|co\.?|company|pty\.?|plc|llp|llc|"
    r"sa|nv|ag|gmbh|bv|sas|sarl|srl|pvt\.?)"
    r"\.?\s*$"
)

def name_short(col):
    """Search-friendly customer name: lowercase, trailing business-entity
    suffixes stripped.  Applied twice to handle stacked endings like 'Pty Ltd'.

    Examples
    --------
    'Bakari Horticulture Ltd'  ->  'bakari horticulture'
    'Databank Group'           ->  'databank'
    'Admin 365'                ->  'admin 365'   (no suffix, unchanged)
    """
    _strip = lambda c: F.lower(F.trim(F.regexp_replace(F.trim(c), _BIZ_SUFFIX, "")))
    return _strip(_strip(col))

def to_num(col):
    """Best-effort numeric cast (strips % and other non-numeric characters)."""
    return F.regexp_replace(col.cast("string"), r"[^0-9.\-]", "").cast("double")

print("Helpers ready. Default lakehouse = ManagedServiceData (dbo).")


StatementMeta(, 4e775791-4963-4bd4-8353-a2cf55f9bf06, 3, Finished, Available, Finished, False)

Helpers ready. Default lakehouse = ManagedServiceData (dbo).


In [ ]:
# Entity: Customer  ->  sem_dim_customer
# One row per customer/tenant, enriched with Inforcer posture, CRM account,
# and Partner Center (pc_customers) fields.

inf = rd("inforcer_tenants").select(
    norm_key("tenantFriendlyName").alias("customer_key"),
    F.col("tenantFriendlyName").alias("inf_name"),
    "clientTenantId", "msTenantId", "tenantDnsName",
    F.col("tenant_type").alias("inf_tenant_type"),
    # NOTE: This is the Inforcer backup/management plan tier (e.g. Premium/Standard),
    # NOT the customer's Microsoft 365 / Copilot licensing (not available in source).
    F.col("licenseSkus").alias("inforcer_plan_tier"),
    to_num(F.col("secureScore")).alias("secure_score"),
    "isBaseline", "lastBackupTimestamp",
)

corr = rd("inforcer_crm_correlation").select(
    "clientTenantId",
    F.col("AccountID").alias("corr_account_id"),
    F.col("CrmAccountName").alias("corr_account_name"),
    F.col("Owner").alias("corr_owner"),
    F.col("OwnerEmail").alias("corr_owner_email"),
    F.col("PrimaryContactEmail").alias("corr_contact_email"),
)

# CRM fallback via the domain-derived customer map (one row per tenant, prefer a CRM match)
w_ctd = Window.partitionBy(norm_key(F.col("TenantName"))).orderBy(
    F.when(F.lower(F.col("CrmMatch").cast("string")).isin("true", "1", "yes"), 0).otherwise(1),
    F.col("BuiltAt").desc_nulls_last(),
)
ctd = (rd("customer_tenant_domains")
    .withColumn("customer_key", norm_key(F.col("TenantName")))
    .withColumn("_rn", F.row_number().over(w_ctd))
    .filter(F.col("_rn") == 1)
    .select(
        "customer_key",
        F.col("TenantName").alias("ctd_name"),
        F.col("TenantDomain").alias("ctd_dns"),
        F.col("CrmAccountID").alias("ctd_account_id"),
        F.col("CrmAccountName").alias("ctd_account_name"),
        F.col("RelationshipType").alias("relationship_type"),
        F.col("CrmPrimaryContactEmail").alias("ctd_contact_email"),
        F.col("Seller").alias("ctd_owner"),
        F.col("SellerEmail").alias("ctd_owner_email"),
        F.col("Territory").alias("territory"),
        F.col("AccountStatus").alias("account_status"),
    ))

# ── Partner Center enrichment ────────────────────────────────────────────────
pc = (rd("pc_customers")
    .select(
        F.col("tenant_id").alias("pc_tenant_id"),
        norm_key(F.col("company_name")).alias("pc_name_key"),
        F.col("company_name").alias("pc_company_name"),
        F.col("domain").alias("pc_domain"),
        F.col("country").alias("pc_country"),
        F.col("city").alias("pc_city"),
        F.col("state").alias("pc_state"),
        F.col("billing_email").alias("pc_billing_email"),
        F.col("billing_contact_name").alias("pc_billing_contact"),
        F.col("relationship").alias("pc_csp_relationship"),
        F.col("allow_delegated_admin").alias("pc_allow_delegated_admin"),
    ))

# ── Graph API tenants (non-PC, consent-based via M365GraphIngestion) ─────────
# Customers onboarded via Microsoft Graph — not in Partner Center or Inforcer.
# Seeded into the base universe so they appear in the dimension with their
# M365 tenant ID and default domain, enabling cross-join with SKU/promo data.
try:
    graph = (rd("graph_tenant_skus")
        .filter(F.col("consent_status") == "ok")
        .select(
            F.col("tenant_id").alias("graph_tenant_id"),
            F.col("company_name").alias("graph_company_name"),
            F.col("domain").alias("graph_domain"),
            norm_key(F.col("company_name")).alias("graph_name_key"),
        )
        .dropDuplicates(["graph_tenant_id"])
    )
except Exception:
    graph = spark.createDataFrame(
        [], "graph_tenant_id string, graph_company_name string, graph_domain string, graph_name_key string"
    )

# Canonical tenant universe: all assessment families + Inforcer + CRM-domain map + PC + Graph
def names_from(table):
    return rd(table).select(
        norm_key(F.col("tenant_name")).alias("customer_key"),
        F.col("tenant_name").alias("src_name"),
    )

base = (
    names_from("security_assessment_assessments")
    .unionByName(names_from("copilot_readiness_assessments"))
    .unionByName(names_from("copilot_assessment_assessments"))
    .unionByName(inf.select("customer_key", F.col("inf_name").alias("src_name")))
    .unionByName(ctd.select("customer_key", F.col("ctd_name").alias("src_name")))
    # Seed PC-only tenants
    .unionByName(pc.select(F.col("pc_name_key").alias("customer_key"),
                           F.col("pc_company_name").alias("src_name")))
    # Seed Graph-only tenants (non-PC, non-Inforcer)
    .unionByName(graph.select(F.col("graph_name_key").alias("customer_key"),
                              F.col("graph_company_name").alias("src_name")))
    .filter(F.col("customer_key").isNotNull() & (F.col("customer_key") != ""))
)
w_name = Window.partitionBy("customer_key").orderBy(F.length("src_name").desc())
base = (base.withColumn("_rn", F.row_number().over(w_name))
             .filter(F.col("_rn") == 1)
             .select("customer_key", F.col("src_name").alias("tenant_name")))

sem_dim_customer = (
    base
    .join(inf, "customer_key", "left")
    .join(corr, "clientTenantId", "left")
    .join(ctd, "customer_key", "left")
    # Enrich with Partner Center data — join on clientTenantId (exact tenant GUID match)
    .join(pc, F.col("clientTenantId") == F.col("pc_tenant_id"), "left")
    # Enrich with Graph API tenants — join on customer_key for Graph-only tenants,
    # fall back to clientTenantId match for tenants that are in both Inforcer and Graph
    .join(graph, F.col("customer_key") == F.col("graph_name_key"), "left")
    .select(
        "customer_key", "tenant_name", "clientTenantId", "msTenantId",
        F.coalesce("tenantDnsName", "ctd_dns", "pc_domain", "graph_domain").alias("tenant_dns"),
        F.coalesce("inf_tenant_type", F.lit("Assessment")).alias("tenant_type"),
        "inforcer_plan_tier", "secure_score", "isBaseline", "lastBackupTimestamp",
        F.coalesce("corr_account_id", "ctd_account_id").alias("crm_account_id"),
        F.coalesce("corr_account_name", "ctd_account_name").alias("crm_account_name"),
        "relationship_type",
        F.coalesce("corr_owner", "ctd_owner").alias("owner"),
        F.coalesce("corr_owner_email", "ctd_owner_email").alias("owner_email"),
        F.coalesce("corr_contact_email", "ctd_contact_email").alias("primary_contact_email"),
        "territory", "account_status",
        # ── Partner Center fields ─────────────────────────────────────────────
        "pc_tenant_id",
        "pc_company_name",
        "pc_domain",
        "pc_country",
        "pc_city",
        "pc_state",
        "pc_billing_email",
        "pc_billing_contact",
        "pc_csp_relationship",
        "pc_allow_delegated_admin",
        # ── Graph API fields (non-PC tenants) ─────────────────────────────────
        "graph_tenant_id",
        "graph_domain",
    )
    .withColumn("has_crm_match",   F.col("crm_account_id").isNotNull())
    .withColumn("has_pc_match",    F.col("pc_tenant_id").isNotNull())
    .withColumn("has_graph_match", F.col("graph_tenant_id").isNotNull())
    .withColumn("built_at", F.current_timestamp())
    .dropDuplicates(["customer_key"])
)

# Search-friendly name
sem_dim_customer = sem_dim_customer.withColumn("short_name", name_short(F.col("tenant_name")))

write_gold(sem_dim_customer, "sem_dim_customer")
display(sem_dim_customer.limit(20))


StatementMeta(, 7f659c60-5fed-4553-ac36-a8c63b641133, 4, Finished, Available, Finished, False)

wrote dbo.sem_dim_customer: 353 rows, 35 cols


SynapseWidget(Synapse.DataFrame, a4e45172-8d2c-409a-b447-d1018a860f83)

In [4]:
# Fact: Assessment  ->  sem_fact_assessment
# One row per assessment run across the three assessment families.

common = ["assessment_id", "tenant_name", "assessment_type", "assessment_name",
          "assessment_date", "assessment_time", "overall_score_pct",
          "passed_count", "failed_count", "warnings_count"]

def asmt(table, family):
    return rd(table).select(*common).withColumn("assessment_family", F.lit(family))

sem_fact_assessment = (
    asmt("security_assessment_assessments", "Security")
    .unionByName(asmt("copilot_readiness_assessments", "CopilotReadiness"))
    .unionByName(asmt("copilot_assessment_assessments", "CopilotAssessment"))
    .withColumn("customer_key", norm_key(F.col("tenant_name")))
    .withColumn("short_name", name_short(F.col("tenant_name")))
    .withColumn("overall_score_pct", to_num(F.col("overall_score_pct")))
    .withColumn("assessment_date_parsed", F.to_date(F.col("assessment_date")))
)

write_gold(sem_fact_assessment, "sem_fact_assessment")
display(sem_fact_assessment.groupBy("assessment_family").count().orderBy("assessment_family"))


StatementMeta(, 22a2e431-4463-49d8-8e6c-c75208803946, 10, Finished, Available, Finished, False)

wrote dbo.sem_fact_assessment: 373 rows, 14 cols


SynapseWidget(Synapse.DataFrame, 4914aa16-43b7-4438-87d5-e50142674547)

In [ ]:
# Fact: Check  ->  sem_fact_check
# One row per control/check result (Security + Copilot Readiness share this schema).
# is_latest_assessment = True marks checks from the most recent run per (tenant, family),
# enabling agents to query current-state checks without cross-run duplicates.

chk_cols = ["check_id", "assessment_id", "tenant_name", "category", "subcategory",
            "category_label", "check_name", "business_rationale", "status",
            "priority", "frameworks", "framework_control", "framework_level"]

def chk(table, family):
    return rd(table).select(*chk_cols).withColumn("check_family", F.lit(family))

# Identify the latest assessment_id per tenant per family from the source headers
def latest_asmt_id(asmt_table):
    w = Window.partitionBy(norm_key(F.col("tenant_name"))).orderBy(
        F.to_date(F.col("assessment_date")).desc_nulls_last(),
        F.col("assessment_time").desc_nulls_last(),
    )
    return (rd(asmt_table)
        .withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .select(F.col("assessment_id").alias("_latest_id")))

latest_ids = (
    latest_asmt_id("security_assessment_assessments")
    .unionByName(latest_asmt_id("copilot_readiness_assessments"))
)

sem_fact_check = (
    chk("security_assessment_checks", "Security")
    .unionByName(chk("copilot_readiness_checks", "CopilotReadiness"))
    .withColumn("customer_key", norm_key(F.col("tenant_name")))
    .withColumn("is_issue", F.lower(F.coalesce(F.col("status"), F.lit(""))).rlike("fail|warn"))
    .join(F.broadcast(latest_ids), F.col("assessment_id") == F.col("_latest_id"), "left")
    .withColumn("is_latest_assessment", F.col("_latest_id").isNotNull())
    .drop("_latest_id")
)

# ── Data quality: strip PDF section-header suffixes from check_name ───────────
#
# The PDF parser sometimes appends a section label to check_name.
# e.g. "Ensure DLP is enabled Data Loss Prevention"  (dirty)
#   vs "Ensure DLP is enabled"                        (clean)
#
# Section headers are detected in two ways:
#   A. Dynamic: short suffixes (≤ 4 words) that appear appended to ≥ 3 distinct
#      base check names — recurring section labels by definition.
#   B. Seed list: known PDF section headers that may appear with only 1–2 bases.
#
# For each dirty name, we pick the LONGEST matching clean base (most specific).
# Dirty names are REPLACED (not deleted) to preserve check counts per family.
# Dedup then removes any exact duplicates created within the same assessment.

# ── Seed: PDF section headers confirmed from the assessment PDF structure ─────
SEED_HEADERS = [
    "Audit", "Devices", "Data Loss Prevention", "Settings", "Policies",
    "Email & Collaboration", "Identity Governance", "Users", "Conditional Access",
    "Authentication Methods", "Application Management", "Sharing", "Meetings",
    "External Identities", "Information Protection", "Group Settings", "Org Settings",
]

seed_df = spark.createDataFrame([(h,) for h in SEED_HEADERS], ["suffix"])

sem_fact_check.createOrReplaceTempView("_chk_raw")

# Step 1: extract all (dirty, base, suffix) candidates where base ≥ 25 chars
all_pairs = spark.sql("""
    SELECT DISTINCT
        a.check_name                                              AS dirty_name,
        b.check_name                                              AS base_name,
        TRIM(SUBSTR(a.check_name, length(b.check_name) + 2))    AS suffix
    FROM (SELECT DISTINCT check_name FROM _chk_raw) a
    JOIN (SELECT DISTINCT check_name FROM _chk_raw) b
      ON a.check_name LIKE concat(b.check_name, ' %')
     AND length(a.check_name) > length(b.check_name)
     AND length(b.check_name) >= 25
""")
all_pairs.createOrReplaceTempView("_pairs")

# Step 2: dynamic section headers (≥ 3 bases, ≤ 4 words)
dynamic_headers = spark.sql("""
    SELECT suffix
    FROM _pairs
    GROUP BY suffix
    HAVING COUNT(DISTINCT base_name) >= 3
       AND SIZE(SPLIT(TRIM(suffix), ' ')) <= 4
""")

# Union with seed list — combined set of validated section headers
all_headers = dynamic_headers.unionByName(seed_df).distinct()
all_headers.createOrReplaceTempView("_section_headers")

# Step 3: build dirty → canonical map using validated section-header suffixes only
# For each dirty name, keep the LONGEST matching base (most specific canonical form)
canonical_map = spark.sql("""
    SELECT dirty_name, clean_name
    FROM (
        SELECT p.dirty_name,
               p.base_name AS clean_name,
               ROW_NUMBER() OVER (PARTITION BY p.dirty_name ORDER BY length(p.base_name) DESC) AS rn
        FROM _pairs p
        JOIN _section_headers s ON s.suffix = p.suffix
    )
    WHERE rn = 1
""")

n_dirty = canonical_map.count()
print(f"Dirty check_name variants to fix: {n_dirty}")
canonical_map.show(20, truncate=False)

canonical_map.createOrReplaceTempView("_canonical_map")

# Step 4: apply replacement
sem_fact_check = spark.sql("""
    SELECT
        COALESCE(m.clean_name, a.check_name) AS check_name,
        a.check_id, a.assessment_id, a.tenant_name, a.category, a.subcategory,
        a.category_label, a.business_rationale, a.status, a.priority,
        a.frameworks, a.framework_control, a.framework_level,
        a.check_family, a.customer_key, a.is_issue, a.is_latest_assessment
    FROM _chk_raw a
    LEFT JOIN _canonical_map m ON m.dirty_name = a.check_name
""")

for v in ["_chk_raw", "_pairs", "_section_headers", "_canonical_map"]:
    spark.catalog.dropTempView(v)

# Add search-friendly name (inherited from tenant_name, suffixes stripped)
sem_fact_check = sem_fact_check.withColumn("short_name", name_short(F.col("tenant_name")))

# Step 5: dedup — after renaming, some rows may become exact duplicates
# within the same assessment (clean + dirty for the same check in the same run)
sem_fact_check = sem_fact_check.dropDuplicates(
    ["tenant_name", "assessment_id", "check_name", "category", "status"]
)

write_gold(sem_fact_check, "sem_fact_check")
display(sem_fact_check.groupBy("check_family", "status", "is_latest_assessment")
        .count().orderBy("check_family", "status", "is_latest_assessment"))


StatementMeta(, 22a2e431-4463-49d8-8e6c-c75208803946, 11, Finished, Available, Finished, False)

Dirty check_name variants to fix: 108
+--------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------+
|dirty_name                                                                                  |clean_name                                                                         |
+--------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------+
|Check SharePoint sharing level at organisation level Policies                               |Check SharePoint sharing level at organisation level                               |
|Check SharePoint sharing level per site Sharing                                             |Check SharePoint sharing level per site                                            |
|Check if external sharing is disabled Policies                    

SynapseWidget(Synapse.DataFrame, d6418444-c95f-4b2f-8daa-d59c2f2e68cb)

In [ ]:
# Fact: Security Posture  ->  sem_fact_posture
# One snapshot row per customer: secure score + latest score per assessment family.

def latest_score(table, out_col):
    w = Window.partitionBy(norm_key(F.col("tenant_name"))).orderBy(
        F.to_date(F.col("assessment_date")).desc_nulls_last(),
        F.col("assessment_time").desc_nulls_last(),
    )
    return (rd(table)
        .withColumn("customer_key", norm_key(F.col("tenant_name")))
        .withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .select("customer_key", to_num(F.col("overall_score_pct")).alias(out_col)))

sec = latest_score("security_assessment_assessments", "latest_security_score")
rdy = latest_score("copilot_readiness_assessments", "latest_readiness_score")
cop = latest_score("copilot_assessment_assessments", "latest_copilot_score")

sem_fact_posture = (
    rd("sem_dim_customer").select("customer_key", "tenant_name", "short_name", "secure_score")
    .join(sec, "customer_key", "left")
    .join(rdy, "customer_key", "left")
    .join(cop, "customer_key", "left")
    .withColumn("security_exposure", F.lit(100.0) - F.col("secure_score"))
    # Copilot readiness is low across the board (0-71, avg ~27), and no M365/Copilot
    # license data exists. So "adoption opportunity" is reframed as an ENABLEMENT gap:
    # the further below readiness, the greater the managed-service enablement opportunity.
    .withColumn("readiness_gap", F.lit(100.0) - F.col("latest_readiness_score"))
    .withColumn(
        "enablement_opportunity",
        F.when(F.col("latest_readiness_score").isNull(), F.lit("Unknown"))
         .when(F.col("latest_readiness_score") < 30, F.lit("High"))
         .when(F.col("latest_readiness_score") < 50, F.lit("Medium"))
         .otherwise(F.lit("Low")),
    )
    .withColumn("built_at", F.current_timestamp())
)

write_gold(sem_fact_posture, "sem_fact_posture")
display(sem_fact_posture.limit(20))


StatementMeta(, 22a2e431-4463-49d8-8e6c-c75208803946, 12, Finished, Available, Finished, False)

wrote dbo.sem_fact_posture: 353 rows, 11 cols


SynapseWidget(Synapse.DataFrame, b7a29018-f9e2-420e-abac-c5906312c86b)

In [ ]:
# Recommendation catalog + tenant opportunities
#   sem_dim_recommendation : the catalog (talking points / SKUs)
#   sem_fact_recommendation: managed-service opportunities = failed checks matched to catalog

sem_dim_recommendation = rd("recommendation_catalog")
write_gold(sem_dim_recommendation, "sem_dim_recommendation")

cat = (rd("recommendation_catalog")
    .filter(F.lower(F.coalesce(F.col("active").cast("string"), F.lit("true"))).isin("true", "1", "yes"))
    .select("rule_id", "assessment_type", "match_category", "match_keyword",
            "recommendation_sku", "recommendation_type", "talking_point",
            to_num(F.col("impact_weight")).alias("impact_weight")))

issues = (rd("sem_fact_check").filter(F.col("is_issue") & F.col("is_latest_assessment"))
          .select("customer_key", "check_family", "category", "subcategory", "check_name"))

# Match when the catalog keyword appears in the check category / subcategory / name
matched = issues.crossJoin(F.broadcast(cat)).where(
    (F.col("match_keyword").isNotNull()) & (F.col("match_keyword") != "") & (
        F.lower(F.coalesce(F.col("category"), F.lit(""))).contains(F.lower(F.col("match_keyword"))) |
        F.lower(F.coalesce(F.col("subcategory"), F.lit(""))).contains(F.lower(F.col("match_keyword"))) |
        F.lower(F.coalesce(F.col("check_name"), F.lit(""))).contains(F.lower(F.col("match_keyword")))
    )
)

cust_lookup = rd("sem_dim_customer").select("customer_key", "tenant_name", "short_name")

sem_fact_recommendation = (
    matched.groupBy("customer_key", "rule_id", "recommendation_sku",
                    "recommendation_type", "talking_point", "impact_weight", "assessment_type")
    .agg(F.count(F.lit(1)).alias("matched_issue_count"))
    .join(F.broadcast(cust_lookup), "customer_key", "left")
    .withColumn("built_at", F.current_timestamp())
)

write_gold(sem_fact_recommendation, "sem_fact_recommendation")
display(sem_fact_recommendation.orderBy(F.col("matched_issue_count").desc()).limit(20))


StatementMeta(, 04969137-c74e-4fe7-a601-ecbb7642bb3e, 5, Finished, Available, Finished, False)

wrote dbo.sem_dim_recommendation: 49 rows, 11 cols
wrote dbo.sem_fact_recommendation: 1,574 rows, 11 cols


SynapseWidget(Synapse.DataFrame, db9f9075-d949-4b7f-b5aa-ef9db75a5fc5)

In [17]:
# Validation: row counts + relationship integrity
for t in ["sem_dim_customer", "sem_fact_assessment", "sem_fact_check",
          "sem_fact_posture", "sem_dim_recommendation", "sem_fact_recommendation"]:
    print(f"{t:26s} -> {rd(t).count():>7,} rows")

cust_keys = rd("sem_dim_customer").select("customer_key")
print("\nCustomers with a CRM match :", rd("sem_dim_customer").filter(F.col("has_crm_match")).count())
print("Assessments w/o a customer :", rd("sem_fact_assessment").join(cust_keys, "customer_key", "left_anti").count())
print("Checks w/o a customer      :", rd("sem_fact_check").join(cust_keys, "customer_key", "left_anti").count())
print("Opportunities (rows)       :", rd("sem_fact_recommendation").count())


StatementMeta(, 7f659c60-5fed-4553-ac36-a8c63b641133, 5, Finished, Available, Finished, False)

sem_dim_customer           ->     353 rows
sem_fact_assessment        ->     373 rows
sem_fact_check             ->  20,628 rows
sem_fact_posture           ->     353 rows
sem_dim_recommendation     ->      49 rows
sem_fact_recommendation    ->   1,574 rows

Customers with a CRM match : 4
Assessments w/o a customer : 0
Checks w/o a customer      : 0
Opportunities (rows)       : 1574


In [ ]:
# ============================================================================
# REGISTER GRAPH TABLES IN METASTORE
# ============================================================================
# graph_tenant_skus and graph_tenant_promo_signals are written via ABFSS path
# in M365GraphIngestion. Register them in the metastore so the SQL endpoint
# (and CheckDetailAgent AI Skill) can query them.

GRAPH_TABLES = ["graph_tenant_skus", "graph_tenant_promo_signals"]

for t in GRAPH_TABLES:
    try:
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS dbo.{t}
            USING DELTA
            LOCATION 'Tables/dbo/{t}'
        """)
        cnt = spark.table(f"dbo.{t}").count()
        print(f"  ✓ dbo.{t} — registered ({cnt:,} rows)")
    except Exception as e:
        print(f"  ✗ dbo.{t} — {e}")


StatementMeta(, 4e775791-4963-4bd4-8353-a2cf55f9bf06, 4, Submitted, Running, Running, True)

  ✓ dbo.graph_tenant_skus — registered (20 rows)


## Reference — Semantic model relationships & DAX measures

Apply these when building the Power BI semantic model over the `sem_*` tables
(and generate the Fabric IQ ontology from that model).

### Relationships (single-direction, one-to-many unless noted)

| From (one) | Column | To (many) | Column | Cardinality |
|------------|--------|-----------|--------|-------------|
| `sem_dim_customer` | `customer_key` | `sem_fact_assessment` | `customer_key` | 1 → * |
| `sem_dim_customer` | `customer_key` | `sem_fact_check` | `customer_key` | 1 → * |
| `sem_dim_customer` | `customer_key` | `sem_fact_posture` | `customer_key` | 1 → 1 |
| `sem_dim_customer` | `customer_key` | `sem_fact_recommendation` | `customer_key` | 1 → * |
| `sem_fact_assessment` | `assessment_id` | `sem_fact_check` | `assessment_id` | 1 → * |
| `sem_dim_recommendation` | `rule_id` | `sem_fact_recommendation` | `rule_id` | 1 → * |

`sem_dim_customer` and `sem_dim_recommendation` are dimensions; the `sem_fact_*` tables are facts.

### DAX measures (create on the fact tables)

```dax
-- Posture
Copilot Readiness =
AVERAGE ( sem_fact_posture[latest_readiness_score] )

Security Score =
AVERAGE ( sem_fact_posture[secure_score] )

Security Exposure =
AVERAGE ( sem_fact_posture[security_exposure] )   -- = 100 - secure_score

Copilot Assessment Score =
AVERAGE ( sem_fact_posture[latest_copilot_score] )

-- Assessment volume
Assessment Count =
DISTINCTCOUNT ( sem_fact_assessment[assessment_id] )

Avg Overall Score =
AVERAGE ( sem_fact_assessment[overall_score_pct] )

-- Checks
Open Issues =
CALCULATE ( COUNTROWS ( sem_fact_check ), sem_fact_check[is_issue] = TRUE () )

Failed Check Rate =
DIVIDE ( [Open Issues], COUNTROWS ( sem_fact_check ) )

-- Opportunities
Open Recommendations =
DISTINCTCOUNT ( sem_fact_recommendation[rule_id] )

Opportunity Weight =
SUMX (
    sem_fact_recommendation,
    sem_fact_recommendation[impact_weight] * sem_fact_recommendation[matched_issue_count]
)

-- Customer coverage
Customers =
DISTINCTCOUNT ( sem_dim_customer[customer_key] )

CRM Matched Customers =
CALCULATE ( [Customers], sem_dim_customer[has_crm_match] = TRUE () )

-- Business flag: high-value Copilot adoption opportunity
Adoption Opportunity =
VAR Readiness = [Copilot Readiness]
VAR HasCopilot =
    CONTAINSSTRING (
        CALCULATE ( CONCATENATEX ( sem_dim_customer, sem_dim_customer[licenseSkus], "|" ) ),
        "COPILOT"
    )
RETURN
    IF ( Readiness > 80 && NOT HasCopilot, "High", "Low" )
```


In [9]:
# Inspect gold-table schemas — confirm exact column names for the DAX measures
sem_tables = [
    "sem_dim_customer",
    "sem_dim_recommendation",
    "sem_fact_assessment",
    "sem_fact_check",
    "sem_fact_posture",
    "sem_fact_recommendation",
]

for t in sem_tables:
    cols = spark.table(f"dbo.{t}").columns
    print(f"{t} ({len(cols)} cols):")
    for c in cols:
        print(f"    {c}")
    print()


StatementMeta(, f98b157a-b62d-4dfb-9cd5-4c4b1ab899ab, 3, Finished, Available, Finished, False)

sem_dim_customer (20 cols):
    customer_key
    tenant_name
    clientTenantId
    msTenantId
    tenant_dns
    tenant_type
    licenseSkus
    secure_score
    isBaseline
    lastBackupTimestamp
    crm_account_id
    crm_account_name
    relationship_type
    owner
    owner_email
    primary_contact_email
    territory
    account_status
    has_crm_match
    built_at

sem_dim_recommendation (11 cols):
    rule_id
    assessment_type
    match_category
    match_keyword
    recommendation_sku
    recommendation_type
    talking_point
    impact_weight
    active
    created_at
    updated_at

sem_fact_assessment (13 cols):
    assessment_id
    tenant_name
    assessment_type
    assessment_name
    assessment_date
    assessment_time
    overall_score_pct
    passed_count
    failed_count
    warnings_count
    assessment_family
    customer_key
    assessment_date_parsed

sem_fact_check (16 cols):
    check_id
    assessment_id
    tenant_name
    category
    subcategory
    ca

In [ ]:
# Ground-truth check: high Copilot readiness + no Copilot license
from pyspark.sql import functions as F

cust = spark.table("dbo.sem_dim_customer").select("customer_key", "tenant_name", "licenseSkus")
post = spark.table("dbo.sem_fact_posture").select("customer_key", "latest_readiness_score")

j = cust.join(post, "customer_key", "left")

# 1) How is licenseSkus actually stored? (format matters for the "no Copilot" filter)
print("=== Sample licenseSkus values ===")
j.select("licenseSkus").where(F.col("licenseSkus").isNotNull()).distinct().show(20, truncate=False)

# 2) Readiness distribution
print("=== Readiness score coverage ===")
j.select(
    F.count("*").alias("total"),
    F.count("latest_readiness_score").alias("has_readiness"),
    F.sum(F.when(F.col("latest_readiness_score") >= 80, 1).otherwise(0)).alias("readiness_ge_80"),
).show()

# 3) High readiness AND licenseSkus does not mention copilot
hi = j.filter(F.col("latest_readiness_score") >= 80)
no_cop = hi.filter(~F.lower(F.coalesce(F.col("licenseSkus"), F.lit(""))).contains("copilot"))
print(f"High readiness (>=80): {hi.count()}   |   of those WITHOUT 'copilot' in licenseSkus: {no_cop.count()}")

no_cop.select("tenant_name", "latest_readiness_score", "licenseSkus").orderBy(
    F.col("latest_readiness_score").desc()
).show(20, truncate=False)


StatementMeta(, 9d0be590-1651-48b6-84a0-9a26b637763a, 6, Finished, Available, Finished, False)

=== Sample licenseSkus values ===
+-----------+
|licenseSkus|
+-----------+
|PREMIUM    |
+-----------+

=== Readiness score coverage ===
+-----+-------------+---------------+
|total|has_readiness|readiness_ge_80|
+-----+-------------+---------------+
|  108|           97|              0|
+-----+-------------+---------------+

High readiness (>=80): 0   |   of those WITHOUT 'copilot' in licenseSkus: 0
+-----------+----------------------+-----------+
|tenant_name|latest_readiness_score|licenseSkus|
+-----------+----------------------+-----------+
+-----------+----------------------+-----------+



In [2]:
# Root-cause diagnostics for readiness scale + license SKU source
from pyspark.sql import functions as F

# 1) Readiness score distribution (pick a correct threshold / verify scale)
post = spark.table("dbo.sem_fact_posture")
print("=== latest_readiness_score distribution ===")
post.select(
    F.min("latest_readiness_score").alias("min"),
    F.expr("percentile_approx(latest_readiness_score, 0.5)").alias("median"),
    F.avg("latest_readiness_score").alias("avg"),
    F.expr("percentile_approx(latest_readiness_score, 0.9)").alias("p90"),
    F.max("latest_readiness_score").alias("max"),
).show()

# 2) Where do Copilot licenses actually live? Inspect inforcer_tenant_licenses SKUs
lic = spark.table("dbo.inforcer_tenant_licenses")
print("=== inforcer_tenant_licenses columns ===", lic.columns)
print("=== distinct SKUs mentioning 'copilot' ===")
lic.select("sku").where(F.lower(F.col("sku")).contains("copilot")).distinct().show(50, truncate=False)
print("=== top SKUs overall ===")
lic.groupBy("sku").count().orderBy(F.col("count").desc()).show(30, truncate=False)

# 3) Confirm licenseSkus in the source tenants table
ten = spark.table("dbo.inforcer_tenants")
print("=== inforcer_tenants.licenseSkus sample ===")
ten.select("tenantFriendlyName", "licenseSkus").show(10, truncate=False)


StatementMeta(, 9d0be590-1651-48b6-84a0-9a26b637763a, 7, Finished, Available, Finished, False)

=== latest_readiness_score distribution ===
+---+------+-----------------+----+----+
|min|median|              avg| p90| max|
+---+------+-----------------+----+----+
|0.0|  29.0|26.95876288659794|43.0|71.0|
+---+------+-----------------+----+----+

=== inforcer_tenant_licenses columns === ['parent_id', 'sku']
=== distinct SKUs mentioning 'copilot' ===
+---+
|sku|
+---+
+---+

=== top SKUs overall ===
+-------+-----+
|sku    |count|
+-------+-----+
|PREMIUM|15   |
+-------+-----+

=== inforcer_tenants.licenseSkus sample ===
+------------------------------------+-----------+
|tenantFriendlyName                  |licenseSkus|
+------------------------------------+-----------+
|Reliance Software Design            |PREMIUM    |
|KINETIC TOURS COMPANY LIMITED       |PREMIUM    |
|DANG Lifestyle Inc                  |PREMIUM    |
|Ghana College of Nurses and Midwives|PREMIUM    |
|University of Gold Coast            |PREMIUM    |
|Ebony Oil & Gas Limited             |PREMIUM    |
|RG Estate 

In [4]:
# Ground-truth: does "Bakari Horticulture" actually have failed CopilotReadiness checks?
# Confirms whether the agent's "no data" is real or a retrieval miss.
chk = spark.table("dbo.sem_fact_check").filter(F.lower(F.col("tenant_name")).contains("bakari"))

print("Distinct tenant_name values matching 'bakari':")
chk.select("tenant_name").distinct().show(truncate=False)

print("Counts by check_family / status / is_issue:")
(chk.groupBy("check_family", "status", "is_issue")
    .count().orderBy("check_family", "status").show(50, truncate=False))

print("Failed/Warning CopilotReadiness checks (sample):")
(chk.filter((F.col("check_family") == "CopilotReadiness") & (F.col("is_issue") == True))
    .select("check_name", "category", "status", "priority")
    .show(20, truncate=False))


StatementMeta(, 78b0f01c-730f-4e1f-882f-a5f1085da6f6, 11, Finished, Available, Finished, False)

Distinct tenant_name values matching 'bakari':
+-------------------+
|tenant_name        |
+-------------------+
|Bakari Horticulture|
+-------------------+

Counts by check_family / status / is_issue:
+----------------+-------+--------+-----+
|check_family    |status |is_issue|count|
+----------------+-------+--------+-----+
|CopilotReadiness|Failed |true    |32   |
|CopilotReadiness|Passed |false   |8    |
|CopilotReadiness|Warning|true    |2    |
|Security        |Failed |true    |162  |
|Security        |Passed |false   |30   |
|Security        |Unknown|false   |2    |
+----------------+-------+--------+-----+

Failed/Warning CopilotReadiness checks (sample):
+---------------------------------------------------------------------------------------------+----------------------------+------+--------+
|check_name                                                                                   |category                    |status|priority|
+---------------------------------------------

In [12]:
# Diagnostic: exact column names + Bakari's posture values
spark.sql("DESCRIBE TABLE dbo.sem_fact_posture").show(30, truncate=False)

print("--- Bakari posture ---")
spark.sql("""
    SELECT tenant_name, short_name,
           latest_security_score,
           latest_readiness_score,
           latest_copilot_score,
           enablement_opportunity,
           readiness_gap,
           security_exposure
    FROM dbo.sem_fact_posture
    WHERE lower(short_name) LIKE '%bakari%'
""").show(truncate=False)


StatementMeta(, 25af7f8b-b873-4ddc-a251-75b14e9a3683, 4, Finished, Available, Finished, False)

+----------------------+---------+-------+
|col_name              |data_type|comment|
+----------------------+---------+-------+
|customer_key          |string   |NULL   |
|tenant_name           |string   |NULL   |
|short_name            |string   |NULL   |
|secure_score          |double   |NULL   |
|latest_security_score |double   |NULL   |
|latest_readiness_score|double   |NULL   |
|latest_copilot_score  |double   |NULL   |
|security_exposure     |double   |NULL   |
|readiness_gap         |double   |NULL   |
|enablement_opportunity|string   |NULL   |
|built_at              |timestamp|NULL   |
+----------------------+---------+-------+

--- Bakari posture ---
+-----------------------+-------------------+---------------------+----------------------+--------------------+----------------------+-------------+-----------------+
|tenant_name            |short_name         |latest_security_score|latest_readiness_score|latest_copilot_score|enablement_opportunity|readiness_gap|security_exposur

In [3]:
# Ground-truth: does "KudiPlus Resources" have CopilotReadiness checks in sem_fact_check?
# Diagnoses why the CheckDetailAgent returns 0 rows for KudiPlus.

print("=== Exact tenant_name values containing 'kudi' ===")
spark.table("dbo.sem_fact_check").filter(
    F.lower(F.col("tenant_name")).contains("kudi")
).select("tenant_name").distinct().show(truncate=False)

print("=== Check counts for KudiPlus by family/status ===")
spark.table("dbo.sem_fact_check").filter(
    F.lower(F.col("tenant_name")).contains("kudi")
).groupBy("check_family", "status").count().orderBy("check_family", "status").show(truncate=False)

print("=== Sample CopilotReadiness Failed checks for KudiPlus ===")
spark.table("dbo.sem_fact_check").filter(
    F.lower(F.col("tenant_name")).contains("kudi") &
    (F.col("check_family") == "CopilotReadiness") &
    (F.col("status").isin("Failed", "Warning"))
).select("tenant_name", "check_name", "category", "status", "priority").show(20, truncate=False)

# Also check sem_fact_posture to confirm KudiPlus has a readiness score
print("=== KudiPlus posture row ===")
spark.table("dbo.sem_fact_posture").join(
    spark.table("dbo.sem_dim_customer").filter(F.lower(F.col("tenant_name")).contains("kudi")).select("customer_key", "tenant_name"),
    "customer_key"
).show(truncate=False)


StatementMeta(, b71388d6-2a27-4cdc-8506-330348463da9, 8, Finished, Available, Finished, False)

=== Exact tenant_name values containing 'kudi' ===
+------------------+
|tenant_name       |
+------------------+
|KudiPlus Resources|
+------------------+

=== Check counts for KudiPlus by family/status ===
+----------------+-------+-----+
|check_family    |status |count|
+----------------+-------+-----+
|CopilotReadiness|Failed |42   |
|Security        |Failed |176  |
|Security        |Passed |12   |
|Security        |Unknown|6    |
+----------------+-------+-----+

=== Sample CopilotReadiness Failed checks for KudiPlus ===
+------------------+---------------------------------------------------------------------------------------------+----------------------------+------+--------+
|tenant_name       |check_name                                                                                   |category                    |status|priority|
+------------------+---------------------------------------------------------------------------------------------+----------------------------+-----

In [4]:
# Verify is_latest_assessment flag for KudiPlus — confirms which checks the agent should return
chk = spark.table("dbo.sem_fact_check")
asmt = spark.table("dbo.sem_fact_assessment")

print("=== is_latest_assessment distribution for KudiPlus CopilotReadiness ===")
(chk.filter(F.lower(F.col("tenant_name")).contains("kudi") &
            (F.col("check_family") == "CopilotReadiness"))
    .groupBy("is_latest_assessment", "assessment_id").count()
    .join(asmt.select("assessment_id", "assessment_date"), "assessment_id", "left")
    .orderBy("is_latest_assessment", "assessment_date")
    .show(truncate=False))

print("=== High-priority Failed CopilotReadiness checks WHERE is_latest_assessment=True ===")
(chk.filter(F.lower(F.col("tenant_name")).contains("kudi") &
            (F.col("check_family") == "CopilotReadiness") &
            (F.col("status").isin("Failed", "Warning")) &
            (F.col("is_latest_assessment") == True) &
            (F.col("priority") == "High"))
    .select("check_name", "category", "status", "priority")
    .orderBy("check_name")
    .show(20, truncate=False))


StatementMeta(, b71388d6-2a27-4cdc-8506-330348463da9, 9, Finished, Available, Finished, False)

=== is_latest_assessment distribution for KudiPlus CopilotReadiness ===
+----------------+--------------------+-----+---------------+
|assessment_id   |is_latest_assessment|count|assessment_date|
+----------------+--------------------+-----+---------------+
|43c8aefebadb02ce|false               |21   |2026-06-02     |
|ae19485909f689a2|true                |21   |2026-06-02     |
+----------------+--------------------+-----+---------------+

=== High-priority Failed CopilotReadiness checks WHERE is_latest_assessment=True ===
+-----------------------------------------------------------------+----------------------------+------+--------+
|check_name                                                       |category                    |status|priority|
+-----------------------------------------------------------------+----------------------------+------+--------+
|Enable Conditional Access policies to block legacy authentication|Identity & Access Management|Failed|High    |
|Ensure 'AuditBypa